[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/17_mlops_monitoring/17_proyecto_pipeline_ct.ipynb)

# Proyecto 17 · Pipeline de entrenamiento continuo con MLflow + Prefect + Evidently

**Nivel:** 🔴 Experto · **Duración:** 4–6 h · **GPU:** no necesaria · **Unidades Colab:** 1–2 (CPU)

## 🏢 Contexto de negocio

El servicio de CineMatch del proyecto 16 está en producción. Tres meses después, el *play rate* de la home ha caído un
4 % y nadie sabe por qué: el modelo se entrenó una vez y nunca más. La VP de producto te pide un **sistema de
entrenamiento continuo** que (a) bloquee datos corruptos, (b) detecte drift, (c) reentrene solo cuando haga falta —el
cómputo cuesta—, (d) promueva modelos solo si son mejores y (e) permita **rollback en segundos**.

Simularás ~15–20 años de producción con MovieLens (cada año = un periodo).

## 📦 Dataset

**MovieLens latest-small** (GroupLens, 100 836 valoraciones, 610 usuarios, 9 742 películas, 1996–2018) —
https://files.grouplens.org/datasets/movielens/ml-latest-small.zip. Con `SCALE = "full"`: **ML-25M** (25M valoraciones,
1995–2019). Fallback sintético con drift si no hay red.

## ✅ Entregables y rúbrica (100 puntos)

| # | Entregable | Criterio objetivo | Puntos |
|---|---|---|---|
| E1 | Esquema Pandera `SCHEMA` | acepta **todos** los lotes anuales reales y rechaza los 3 lotes corruptos de prueba | 15 |
| E2 | `drift_report(model, batch, ref_batch)` | devuelve `oov`, `js_genre`, `evidently_share`, `emb_auc`; gráfico por año para el modelo estático | 20 |
| E3 | Flow de Prefect `ct_pipeline(year)` | tareas ingest → validate → drift → (train → evaluate → register → promote); cada candidato queda registrado en MLflow con métricas; alias `champion`/`challenger` | 25 |
| E4 | Simulación de políticas | NDCG@10 medio de la política **por trigger** ≥ **0,92 ×** el de reentreno anual con ventana, usando **≤ 70 %** de sus reentrenos; gráfico NDCG vs año (estático / anual / trigger) | 25 |
| E5 | Simulacro de incidente | un challenger defectuoso es **rechazado** por la puerta de calidad; un rollback manual del alias se completa y el modelo servido cambia | 15 |

In [ ]:
!pip install -q mlflow prefect evidently pandera scikit-learn scipy pyarrow

In [ ]:
import os, io, re, zipfile, urllib.request, time, json, math, random, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
seed = 42
random.seed(seed); np.random.seed(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# SCALE = "small" → MovieLens latest-small (100K valoraciones, 1996–2018, CPU, minutos)
# SCALE = "full"  → MovieLens-25M (25M valoraciones, 1995–2019; ~2 GB RAM, ~15–25 min en CPU de Colab)
SCALE = "small"
DATA_DIR = "data"; os.makedirs(DATA_DIR, exist_ok=True)
URLS = {"small": ("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml-latest-small"),
        "full": ("https://files.grouplens.org/datasets/movielens/ml-25m.zip", "ml-25m")}


def synthetic_drift(n_items=3000, years=range(1998, 2018), users_per_year=260, d=16, seed=42):
    """Fallback sin red. Catálogo que crece cada año, gustos que se desplazan (taste drift) y preferencia por novedades."""
    rng = np.random.default_rng(seed)
    item_f = rng.normal(0, 1, (n_items, d)) / np.sqrt(d)
    release = np.sort(rng.integers(1980, max(years) + 1, n_items))
    quality = rng.normal(0, 1, n_items)
    G = rng.normal(0, 1, (d, 18)); main_genre = (item_f @ G).argmax(1)
    gnames = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
              "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]
    trend = rng.normal(0, 1, d); trend /= np.linalg.norm(trend)
    rows, uid = [], 0
    for y in years:
        frac = (y - min(years)) / (max(years) - min(years))
        avail = np.where(release <= y)[0]
        for _ in range(users_per_year):
            uid += 1
            u = rng.normal(0, 1, d) + 2.5 * frac * trend
            logit = 3 * item_f[avail] @ u + 0.8 * quality[avail] - 0.12 * (y - release[avail])
            p = np.exp(logit - logit.max()); p /= p.sum()
            n = int(rng.integers(15, 80))
            its = rng.choice(avail, size=min(n, len(avail)), replace=False, p=p)
            aff = item_f[its] @ u
            r = np.clip(np.round((3.2 + 1.5 * aff + 0.3 * quality[its] + rng.normal(0, 0.6, len(its))) * 2) / 2, 0.5, 5)
            t0 = pd.Timestamp(f"{y}-01-01").timestamp()
            ts = t0 + np.sort(rng.uniform(0, 364 * 86400, len(its)))
            rows.append(pd.DataFrame({"user_id": uid, "item_id": its + 1, "rating": r, "ts": ts.astype(int)}))
    ratings = pd.concat(rows, ignore_index=True)
    movies = pd.DataFrame({"item_id": np.arange(1, n_items + 1), "title": [f"Título {i}" for i in range(1, n_items + 1)],
                           "genres": [gnames[g] for g in main_genre], "release_year": release})
    return ratings, movies, "Sintético con drift"


def load_movielens(scale=SCALE):
    url, folder = URLS[scale]
    try:
        path = os.path.join(DATA_DIR, folder + ".zip")
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)
        with zipfile.ZipFile(path) as z:
            r = pd.read_csv(z.open(f"{folder}/ratings.csv"))
            m = pd.read_csv(z.open(f"{folder}/movies.csv"))
        r.columns = ["user_id", "item_id", "rating", "ts"]
        m.columns = ["item_id", "title", "genres"]
        m["release_year"] = pd.to_numeric(m.title.str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce")
        return r, m, folder
    except Exception as e:
        print("⚠️ Sin red o URL caída → datos sintéticos con drift:", repr(e)[:120])
        return synthetic_drift()


ratings, movies, DATASET = load_movielens()
ratings["year"] = pd.to_datetime(ratings.ts, unit="s").dt.year
movies["main_genre"] = movies.genres.str.split("|").str[0]
movies["release_year"] = movies.release_year.fillna(movies.release_year.median())
ratings = ratings.merge(movies[["item_id", "main_genre", "release_year"]], on="item_id", how="left")
ratings["item_age"] = (ratings.year - ratings.release_year).clip(lower=0)
ratings = ratings.sort_values("ts").reset_index(drop=True)
print(DATASET, ratings.shape, "| años:", ratings.year.min(), "→", ratings.year.max())

In [ ]:
from scipy.sparse import csr_matrix


def train_als(df, dim=48, reg=20.0, iters=12, init=None, min_count=3, seed=seed):
    """ALS sin ponderar sobre la matriz implícita binaria (equivale a un SVD regularizado; Hu et al. 2008 añade pesos).

    `init` = modelo previo → *warm start*: los ítems conocidos arrancan con sus factores anteriores.
    Devuelve un dict serializable (es lo que registraremos en MLflow).
    """
    rng = np.random.default_rng(seed)
    cnt = df.item_id.value_counts()
    items = np.sort(cnt[cnt >= min_count].index.to_numpy())
    imap = pd.Series(np.arange(len(items)), index=items)
    d = df[df.item_id.isin(items)].drop_duplicates(["user_id", "item_id"])
    users, uidx = np.unique(d.user_id.to_numpy(), return_inverse=True)
    R = csr_matrix((np.ones(len(d), np.float32), (uidx, imap[d.item_id].to_numpy())), shape=(len(users), len(items)))
    Y = rng.normal(0, 0.01, (len(items), dim)).astype(np.float32)
    if init is not None:
        known = np.isin(items, init["items"])
        prev = pd.Series(np.arange(len(init["items"])), index=init["items"])
        Y[known] = init["Y"][prev[items[known]].to_numpy()]
    I = reg * np.eye(dim, dtype=np.float32)
    for _ in range(iters):
        X = (R @ Y) @ np.linalg.inv(Y.T @ Y + I)
        Y = (R.T @ X) @ np.linalg.inv(X.T @ X + I)
    Y = Y.astype(np.float32)
    pop = cnt.reindex(items).to_numpy().astype(np.float32)
    return {"items": items, "Y": Y, "G": np.linalg.inv(Y.T @ Y + I).astype(np.float32), "pop": pop,
            "trained_until": int(df.ts.max()), "n_train": int(len(d)), "dim": dim, "reg": reg}


def user_vectors(model, hists):
    """Fold-in: x_u = (YᵀY + λI)⁻¹ Y_Hᵀ 1 para cada historial (también sirve para usuarios nunca vistos)."""
    imap = pd.Series(np.arange(len(model["items"])), index=model["items"])
    rows, cols = [], []
    for r, h in enumerate(hists):
        idx = imap.reindex(h).dropna().astype(int).to_numpy()
        rows += [r] * len(idx); cols += list(idx)
    H = csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(hists), len(model["items"])))
    return (H @ model["Y"]) @ model["G"], H


def recommend_batch(model, hists, k=10):
    X, H = user_vectors(model, hists)
    S = X @ model["Y"].T
    S += 1e-6 * np.log1p(model["pop"])[None]          # desempate por popularidad (usuarios sin ítems conocidos)
    S[H.nonzero()] = -np.inf                           # no recomendar lo ya visto
    top = np.argpartition(-S, k, axis=1)[:, :k]
    top = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), 1), 1)
    return model["items"][top], np.take_along_axis(S, top, 1)


def popularity_model(df, window_days=365):
    recent = df[df.ts >= df.ts.max() - window_days * 86400]
    cnt = recent.item_id.value_counts()
    return {"items": cnt.index.to_numpy(), "pop": cnt.to_numpy().astype(np.float32), "trained_until": int(df.ts.max())}


def recommend_pop(model, hists, k=10):
    out = []
    for h in hists:
        s = set(h); out.append([i for i in model["items"][: k + len(h)] if i not in s][:k])
    return np.array(out)


def ndcg_at_k(recs, targets, k=10):
    disc = 1 / np.log2(np.arange(2, k + 2))
    vals = []
    for r, t in zip(recs, targets):
        g = np.isin(r[:k], list(t)).astype(float)
        ideal = disc[: min(k, len(t))].sum()
        vals.append((g * disc).sum() / ideal if ideal else 0.0)
    return float(np.mean(vals))


def eval_sets_by_year(df, min_inter=10, max_users=1500):
    """Para cada año: usuarios activos ese año con ≥ min_inter interacciones. Primera mitad (temporal) = historial
    que se pliega en el modelo (fold-in); segunda mitad = objetivo. Simula 'servir al usuario durante el año'."""
    out = {}
    rng = np.random.default_rng(seed)
    for y, g in df.groupby("year"):
        g = g.sort_values("ts"); cnt = g.user_id.value_counts()
        us = cnt[cnt >= min_inter].index.to_numpy()
        if len(us) > max_users:
            us = rng.choice(us, max_users, replace=False)
        hists, tgts = [], []
        for u, gu in g[g.user_id.isin(us)].groupby("user_id"):
            it = gu.item_id.to_numpy(); h = len(it) // 2
            tg = set(it[h:]) - set(it[:h])
            if tg:
                hists.append(list(it[:h])); tgts.append(tg)
        if len(hists) >= 15:
            out[int(y)] = (hists, tgts)
    return out


EVAL = eval_sets_by_year(ratings)
print("Años evaluables:", list(EVAL)[:3], "…", list(EVAL)[-3:], "| usuarios/año (mediana):",
      int(np.median([len(v[0]) for v in EVAL.values()])))

In [ ]:
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa
from pandera import Check
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


def js_div(ref_cat, cur_cat):
    cats = sorted(set(ref_cat) | set(cur_cat))
    p = pd.Series(ref_cat).value_counts(normalize=True).reindex(cats).fillna(0).to_numpy() + 1e-9
    q = pd.Series(cur_cat).value_counts(normalize=True).reindex(cats).fillna(0).to_numpy() + 1e-9
    m = (p + q) / 2
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


def oov_rate(model, df):
    return float((~df.item_id.isin(model["items"])).mean())


def monitoring_table(df, model):
    pop = pd.Series(model["pop"], index=model["items"])
    act = df.user_id.map(df.user_id.value_counts())
    return pd.DataFrame({"rating": df.rating.astype(float), "item_age": df.item_age.astype(float),
                         "log_item_pop": np.log1p(df.item_id.map(pop).fillna(0)).astype(float),
                         "main_genre": df.main_genre.astype(str), "user_activity": act.astype(float)}).reset_index(drop=True)


def evidently_drift(ref, cur, html_path=None):
    """{columna: (drift?, valor, método)} con Evidently ≥0.7 (o API legacy 0.4.x)."""
    out = {}
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset
        snap = Report([DataDriftPreset()]).run(current_data=cur, reference_data=ref)
        for m in snap.dict()["metrics"]:
            name = m["metric_name"]
            if name.startswith("ValueDrift"):
                col = re.search(r"column=([^,]+)", name).group(1)
                thr = float(re.search(r"threshold=([\d.]+)", name).group(1)); v = float(m["value"])
                out[col] = ((v < thr) if "p_value" in name else (v >= thr), v, name.split("method=")[1].split(",")[0])
    except ImportError:
        from evidently.report import Report
        from evidently.metric_preset import DataDriftPreset
        snap = Report(metrics=[DataDriftPreset()]); snap.run(reference_data=ref, current_data=cur)
        res = snap.as_dict()["metrics"][1]["result"]["drift_by_columns"]
        out = {c: (r["drift_detected"], r["drift_score"], r["stattest_name"]) for c, r in res.items()}
    if html_path:
        snap.save_html(html_path)
    return out


def domain_classifier_auc(A, B):
    X = np.vstack([A, B]); y = np.r_[np.zeros(len(A)), np.ones(len(B))]
    return float(cross_val_score(LogisticRegression(max_iter=500), X, y, cv=3, scoring="roc_auc").mean())


def evaluate(model, t):
    h, tg = EVAL[t]
    return ndcg_at_k(recommend_batch(model, h)[0], tg)


years = sorted(EVAL)
T0 = years[max(1, len(years) // 4)]
YEARS_PROD = [y for y in years if y >= T0]
HP = {"dim": 32, "reg": 5.0}      # configuración razonable del barrido de la lección (re-tunéala si cambias SCALE)
print("Años de producción simulados:", YEARS_PROD[0], "→", YEARS_PROD[-1])

## Paso 1 · Validación de datos (E1)

Define `SCHEMA` (Pandera) para lotes con columnas `user_id, item_id, rating, ts`. Debe rechazar: ratings fuera de
{0,5, 1, …, 5}, timestamps futuros o anteriores a 1995, IDs nulos/no positivos y **> 1 % de duplicados**
`(user_id, item_id, ts)`.

<details><summary>🪜 Pista</summary><code>pa.DataFrameSchema({"rating": pa.Column(float, pa.Check.isin([0.5, 1.0, …, 5.0])), "ts": pa.Column(int, pa.Check.in_range(T_1995, NOW)), "user_id": pa.Column(int, pa.Check.gt(0), nullable=False), …})</code> más un <i>check</i> a nivel de DataFrame: <code>pa.Check(lambda df: df.duplicated(["user_id", "item_id", "ts"]).mean() &lt;= 0.01)</code>. Valida con <code>lazy=True</code> para ver todos los errores a la vez.</details>

In [ ]:
NOW = int(ratings.ts.max()) + 86400
SCHEMA = None   # TODO: pa.DataFrameSchema({...}, checks=[...], coerce=True)


def corrupt_batches(df):
    b1 = df.copy(); b1.loc[b1.sample(frac=0.02, random_state=1).index, "rating"] = 10.0
    b2 = df.copy(); b2.loc[b2.sample(frac=0.01, random_state=2).index, "ts"] = NOW + 10**7
    b3 = pd.concat([df, df.head(max(len(df) // 10, 5))])
    return {"rating_x2": b1, "ts_futuro": b2, "duplicados": b3}


def check_E1(schema):
    cols = ["user_id", "item_id", "rating", "ts"]
    ok_real = all(_valid(schema, ratings[ratings.year == y][cols]) for y in YEARS_PROD)
    bad = corrupt_batches(ratings[ratings.year == YEARS_PROD[0]][cols])
    rejected = {k: not _valid(schema, v) for k, v in bad.items()}
    print(f"E1 · acepta lotes reales: {ok_real} · rechaza corruptos: {rejected}")
    return ok_real and all(rejected.values())


def _valid(schema, df):
    try:
        schema.validate(df, lazy=True); return True
    except pa.errors.SchemaErrors:
        return False


if SCHEMA is None:
    print("⏳ E1 pendiente")
else:
    check_E1(SCHEMA)

## Paso 2 · Informe de drift (E2)

`drift_report(model, batch, ref_batch)` → dict con:
- `oov`: fracción de interacciones de `batch` sobre ítems fuera del vocabulario del modelo;
- `js_genre`: Jensen-Shannon entre la distribución de `main_genre` de `ref_batch` y `batch`;
- `evidently_share`: fracción de columnas de `monitoring_table` con drift según Evidently;
- `emb_auc`: AUC de un clasificador de dominio sobre vectores de usuario (fold-in) de `ref_batch` vs `batch`.

In [ ]:
def drift_report(model: dict, batch: pd.DataFrame, ref_batch: pd.DataFrame) -> dict:
    # TODO
    raise NotImplementedError


try:
    static = train_als(ratings[ratings.year < T0], **HP)
    print(drift_report(static, ratings[ratings.year == YEARS_PROD[-1]], ratings[ratings.year == T0 - 1]))
except NotImplementedError:
    print("⏳ E2 pendiente")

## Paso 3 · Flow de Prefect con MLflow (E3)

Implementa las tareas y el flow. Reglas:
- El servicio carga `models:/cinematch-ct@champion`.
- En el año `year`, el «último lote» es `year-1`. Si no pasa la validación → no se hace nada (y se registra el motivo).
- La **línea base** de drift de cada versión es el informe del primer periodo tras desplegarla; guárdala como *tags*
  de la versión en MLflow (`baseline_oov`, `baseline_js`, …) y no reentrenes ese periodo.
- Si `should_retrain(report, baseline)` → entrenar challenger con ventana de `WINDOW` años hasta `year-2`, evaluar challenger
  y champion en `year-1` (mismo conjunto), registrar el challenger (reentrenado hasta `year-1`) con alias `challenger`
  y promover a `champion` solo si `ndcg_challenger ≥ ndcg_champion`.

<details><summary>🪜 Pista</summary>Cada paso como <code>@task</code> (validar, informe de drift, entrenar, evaluar, registrar) y un <code>@flow ct_pipeline(year)</code> que los encadena con <code>if</code> normales. Para la línea base: <code>client.set_model_version_tag(name, version, "baseline_oov", valor)</code> y léela con <code>client.get_model_version(...).tags</code>. Promoción: <code>client.set_registered_model_alias(name, "champion", version)</code>. <code>should_retrain</code> puede ser «OOV o JS superan la línea base en más de X» (elige X con la simulación del Paso 4).</details>

In [ ]:
WINDOW = 3
MODEL_NAME = "cinematch-ct"


def should_retrain(report: dict, baseline: dict) -> bool:
    # TODO: tu política de trigger: umbrales RELATIVOS a la línea base (oov, js_genre, evidently_share, emb_auc)
    raise NotImplementedError


def ct_pipeline(year: int) -> dict:
    # TODO: flow de Prefect (decorado con @flow) que orqueste las tareas
    raise NotImplementedError

## Paso 4 · Simulación de políticas (E4) y paso 5 · simulacro de incidente (E5)

- `simulate(policy)` para `policy ∈ {"estatico", "anual", "trigger"}` → DataFrame `year, ndcg10, retrained`.
- Simulacro: registra un challenger **defectuoso** (factores aleatorios) y demuestra que tu puerta de calidad lo
  rechaza; después haz un rollback manual del alias y comprueba que el modelo cargado por `@champion` cambia.

---

# ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa a continuación.

In [ ]:
SCHEMA = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, Check.ge(1), nullable=False),
        "item_id": pa.Column(int, Check.ge(1), nullable=False),
        "rating": pa.Column(float, Check.isin(list(np.arange(0.5, 5.01, 0.5))), nullable=False),
        "ts": pa.Column(int, [Check.gt(788_918_400), Check.le(NOW)], nullable=False),
    },
    checks=[Check(lambda d: d.duplicated(["user_id", "item_id", "ts"]).mean() <= 0.01, error="duplicados > 1 %")],
    coerce=True,
)
E1 = check_E1(SCHEMA)

In [ ]:
def drift_report(model, batch, ref_batch, max_users=300):
    mt_ref, mt_cur = monitoring_table(ref_batch, model), monitoring_table(batch, model)
    ev = evidently_drift(mt_ref, mt_cur)
    hist = lambda df: [list(g.sort_values("ts").item_id) for _, g in df.groupby("user_id")][:max_users]
    U_ref, U_cur = user_vectors(model, hist(ref_batch))[0], user_vectors(model, hist(batch))[0]
    return {"oov": oov_rate(model, batch), "js_genre": js_div(ref_batch.main_genre, batch.main_genre),
            "evidently_share": float(np.mean([v[0] for v in ev.values()])),
            "emb_auc": domain_classifier_auc(U_ref, U_cur) if min(len(U_ref), len(U_cur)) >= 10 else 0.5}


static = train_als(ratings[ratings.year < T0], **HP)
ref_b = ratings[ratings.year == T0 - 1]
rep = pd.DataFrame([{"year": y, **drift_report(static, ratings[ratings.year == y], ref_b)} for y in YEARS_PROD])
rep["ndcg10"] = [evaluate(static, y) for y in YEARS_PROD]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for c in ["oov", "js_genre", "evidently_share", "emb_auc"]:
    ax[0].plot(rep.year, rep[c], "o-", ms=3, label=c)
ax[0].legend(); ax[0].set_title("E2 · Drift del modelo estático por año")
ax[1].plot(rep.year, rep.ndcg10, "o-", color="k"); ax[1].set_title("NDCG@10 del modelo estático"); plt.tight_layout(); plt.show()
display(rep.round(3))

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from prefect import flow, task
try:
    from prefect.cache_policies import NO_CACHE
    TK = {"cache_policy": NO_CACHE}
except ImportError:
    TK = {}

mlflow.set_tracking_uri("sqlite:///mlflow_ct.db"); mlflow.set_experiment("cinematch-ct")
client = MlflowClient()


class ALSPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        z = np.load(context.artifacts["model"]); self.m = {k: z[k] for k in z.files}

    def predict(self, context, model_input, params=None):
        return pd.DataFrame({"recs": [list(map(int, r)) for r in recommend_batch(self.m, list(model_input["history"]))[0]]})


def register(model, run_name, params, metrics):
    np.savez("model_ct.npz", **{k: model[k] for k in ("items", "Y", "G", "pop")})
    np.save("model_ct_meta.npy", np.array([model["trained_until"]]))
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params); mlflow.log_metrics(metrics)
        kw = dict(python_model=ALSPyfunc(), artifacts={"model": "model_ct.npz"}, registered_model_name=MODEL_NAME)
        try:
            mlflow.pyfunc.log_model(name="model", **kw)
        except TypeError:
            mlflow.pyfunc.log_model(artifact_path="model", **kw)
    v = max(int(x.version) for x in client.search_model_versions(f"name='{MODEL_NAME}'"))
    client.set_model_version_tag(MODEL_NAME, str(v), "trained_until", str(model["trained_until"]))
    return v


def load_champion():
    info = client.get_model_version_by_alias(MODEL_NAME, "champion")
    m = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion").unwrap_python_model().m
    m["trained_until"] = int(info.tags.get("trained_until", 0))
    return m, int(info.version)


def should_retrain(report, baseline):
    """Deriva 'anormal' = peor que la observada en el primer periodo tras desplegar + margen."""
    return (report["oov"] > baseline["oov"] + 0.05 or report["js_genre"] > baseline["js_genre"] + 0.01
            or report["emb_auc"] > max(baseline["emb_auc"] + 0.10, 0.70))


@task(**TK)
def t_ingest(year):
    return ratings[ratings.year == year]


@task(**TK)
def t_validate(batch):
    return _valid(SCHEMA, batch[["user_id", "item_id", "rating", "ts"]])


@task(**TK)
def t_drift(model, batch):
    ref = ratings[(ratings.ts <= model["trained_until"]) & (ratings.ts > model["trained_until"] - 365 * 86400)]
    return drift_report(model, batch, ref)


@task(**TK)
def t_train(until_year):
    return train_als(ratings[(ratings.year < until_year) & (ratings.year >= until_year - WINDOW)], **HP)


@task(**TK)
def t_eval(model, year):
    return evaluate(model, year)


@flow(name="cinematch-ct-pipeline", log_prints=True)
def ct_pipeline(year: int, force: bool = False, bad_challenger: bool = False) -> dict:
    champ, v_champ = load_champion()
    batch = t_ingest(year - 1)
    if not t_validate(batch):
        return {"year": year, "action": "bloqueado_validacion", "champion": v_champ}
    rep_ = t_drift(champ, batch)
    tags = client.get_model_version(MODEL_NAME, str(v_champ)).tags
    if "baseline_oov" not in tags and not force:          # primer periodo tras desplegar → línea base
        for k in ("oov", "js_genre", "emb_auc"):
            client.set_model_version_tag(MODEL_NAME, str(v_champ), f"baseline_{k}", str(rep_[k]))
        return {"year": year, "action": "linea_base", "champion": v_champ, **rep_}
    baseline = {k: float(tags.get(f"baseline_{k}", rep_[k])) for k in ("oov", "js_genre", "emb_auc")}
    if not (force or should_retrain(rep_, baseline)):
        return {"year": year, "action": "sin_reentreno", "champion": v_champ, **rep_}
    cand = t_train(year - 1)
    if bad_challenger:                                   # simulacro: bug que corrompe los factores
        cand["Y"] = np.random.default_rng(0).normal(0, 1, cand["Y"].shape).astype(np.float32)
    nd_ch, nd_cp = t_eval(cand, year - 1), t_eval(champ, year - 1)
    serve = cand if bad_challenger else t_train(year)
    v = register(serve, f"challenger-{year}", {**HP, "window": WINDOW, "year": year},
                 {"ndcg_val_challenger": nd_ch, "ndcg_val_champion": nd_cp, **rep_})
    client.set_registered_model_alias(MODEL_NAME, "challenger", v)
    promoted = nd_ch >= nd_cp
    if promoted:
        client.set_registered_model_alias(MODEL_NAME, "champion", v)
    print(f"{year}: challenger v{v} {nd_ch:.4f} vs champion v{v_champ} {nd_cp:.4f} → {'PROMOVIDO' if promoted else 'RECHAZADO'}")
    return {"year": year, "action": "promovido" if promoted else "rechazado", "champion": v if promoted else v_champ,
            "challenger": v, **rep_}

In [ ]:
# Estado inicial: champion entrenado hasta T0-1
v_init = register(train_als(ratings[ratings.year < T0], **HP), f"init-{T0}", {**HP, "year": T0}, {})
client.set_registered_model_alias(MODEL_NAME, "champion", v_init)


def run_flow(**kw):
    try:
        return ct_pipeline(**kw)
    except Exception as e:     # Prefect sin servidor temporal → misma lógica sin orquestador
        print("⚠️ Prefect no disponible:", repr(e)[:150]); return ct_pipeline.fn(**kw)


flow_log = []
for y in YEARS_PROD[1:]:
    out = run_flow(year=y)
    champ, _ = load_champion()
    out["ndcg10"] = evaluate(champ, y)                   # lo que el champion consigue sirviendo el año y
    flow_log.append(out)
trig = pd.DataFrame(flow_log)
display(trig[["year", "action", "champion", "oov", "js_genre", "emb_auc", "ndcg10"]].round(3))
E3 = trig.action.isin(["promovido", "rechazado"]).any() and len(client.search_model_versions(f"name='{MODEL_NAME}'")) > 1
print("E3 ·", "✅" if E3 else "❌", "| aliases:", client.get_registered_model(MODEL_NAME).aliases)

In [ ]:
def simulate(policy):
    model = train_als(ratings[ratings.year < T0], **HP); rows = []
    for y in YEARS_PROD:
        re = False
        if y != YEARS_PROD[0] and policy == "anual":
            model, re = train_als(ratings[(ratings.year < y) & (ratings.year >= y - WINDOW)], **HP), True
        rows.append({"policy": policy, "year": y, "ndcg10": evaluate(model, y), "retrained": re})
    return pd.DataFrame(rows)


res = pd.concat([simulate("estatico"), simulate("anual"),
                 pd.DataFrame({"policy": "trigger", "year": [YEARS_PROD[0]] + list(trig.year),
                               "ndcg10": [evaluate(train_als(ratings[ratings.year < T0], **HP), YEARS_PROD[0])] + list(trig.ndcg10),
                               "retrained": [False] + list(trig.action.isin(["promovido", "rechazado"]))})])
summary = res.groupby("policy").agg(ndcg10=("ndcg10", "mean"), reentrenos=("retrained", "sum"))
display(summary)
ratio_q = summary.loc["trigger", "ndcg10"] / summary.loc["anual", "ndcg10"]
ratio_c = summary.loc["trigger", "reentrenos"] / max(summary.loc["anual", "reentrenos"], 1)
print(f"E4 · calidad trigger/anual = {ratio_q:.3f} (≥ 0,92) · reentrenos trigger/anual = {ratio_c:.2f} (≤ 0,70) →",
      "✅" if ratio_q >= 0.92 and ratio_c <= 0.70 else "⚠️ ajusta umbrales de should_retrain")
fig, ax = plt.subplots(figsize=(11, 4))
for p, g in res.groupby("policy"):
    ax.plot(g.year, g.ndcg10.rolling(2, min_periods=1).mean(), "o-", ms=3, label=p)
    rr = g[g.retrained & (p == "trigger")]
    ax.scatter(rr.year, g.ndcg10.rolling(2, min_periods=1).mean()[g.retrained & (p == "trigger")], marker="v", s=60, color="r", zorder=4)
ax.set_title("E4 · NDCG@10 en el tiempo: estático vs anual vs trigger (▼ reentreno)"); ax.set_xlabel("año"); ax.legend(); plt.show()

In [ ]:
# E5 · Simulacro de incidente
champ_before = client.get_model_version_by_alias(MODEL_NAME, "champion").version
drill = run_flow(year=YEARS_PROD[-1], force=True, bad_challenger=True)
rejected = drill["action"] == "rechazado" and client.get_model_version_by_alias(MODEL_NAME, "champion").version == champ_before

# Rollback manual: volvemos a la versión anterior del champion y comprobamos que el modelo servido cambia
versions = sorted(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
prev = [v for v in versions if v < int(champ_before) and v != drill["challenger"]]
t = time.perf_counter()
client.set_registered_model_alias(MODEL_NAME, "champion", prev[-1] if prev else versions[0])
m_after, v_after = load_champion()
rb_ms = 1e3 * (time.perf_counter() - t)
E5 = rejected and str(v_after) != str(champ_before)
print(f"E5 · challenger defectuoso rechazado: {rejected} · rollback v{champ_before} → v{v_after} en {rb_ms:.0f} ms →", "✅" if E5 else "❌")
client.set_registered_model_alias(MODEL_NAME, "champion", champ_before)      # restauramos

## 🚀 Retos extra

1. Sustituye el ALS por el two-tower del módulo 08 con *warm start* desde el checkpoint del champion.
2. Añade un **shadow deploy**: en el flow, antes de promover, sirve el challenger en paralelo al servicio del proyecto 16 y compara distribuciones de score (prediction drift entre modelos).
3. Exporta las métricas del flow a Prometheus (*pushgateway*) y crea la alerta «OOV > 15 % durante 2 periodos» en `reference_stack/prometheus/alerts.yml`.
4. Despliega el flow con `prefect deploy` y un *schedule* cron, o tradúcelo a Airflow (`reference_stack/airflow/`) / Metaflow.
5. Calibra los umbrales de `should_retrain` con una búsqueda en rejilla optimizando `NDCG − λ·reentrenos` (¡en años de validación, no de test!).
6. Implementa **reentreno incremental diario + completo semanal** con datos de ML-25M a nivel de semana.

## 🤔 Reflexión

- ¿Qué parte de tu política depende de etiquetas (y por tanto llega tarde) y cuál no?
- Si el challenger gana offline pero pierde en canary, ¿qué hipótesis investigarías primero?
- ¿Cómo evitarías que el feedback loop haga que cada nuevo modelo «gane» al anterior solo porque se entrena con datos generados por él?
- ¿Qué versionarías junto al modelo para poder reproducir exactamente el champion de hace 6 meses?